<a href="https://colab.research.google.com/github/1rishu0/Computer-Vision/blob/main/Computer_Vision_Masterclass_Image_Segmentation_with_Mask_R_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Masterclass - Image Segmentation with Mask R-CNN

> Note: After an Colab update in December 2022, Tensorflow version 1.x is no longer accessible. So let's use the updated version from the official repository, which has support for the latest versions of Tensorflow.  

> A few more small changes will be necessary (they are demonstrated throughout this Colab)
* Clone from other repository (https://github.com/alsombra/Mask_RCNN-TF2)
* Change from Mask_RCNN (folder name) to Mask_RCNN-TF2
* Add compatibility code (the 5 lines under the title "[ ! ] Compatibility Update")
* Remove the commands that downgrade h5py and tensorflow (they are already removed from this Colab)

## Downloading the repository

In [ ]:
!git clone https://github.com/alsombra/Mask_RCNN-TF2  # updated repository

In [ ]:
!pip install tensorflow==2.15.0

In [ ]:
!pip install numpy==1.26.0

> **Important**: You need to select the option 'Runtime > Restart session'

In [ ]:
%cd Mask_RCNN-TF2

In [ ]:
pwd

In [ ]:
!pip install -r requirements.txt

In [ ]:
!python setup.py install

In [ ]:
%cd ..

In [ ]:
pwd

## Importing the libraries

In [ ]:
import os
import sys
import cv2
import numpy as np
import skimage.io
from google.colab.patches import cv2_imshow
import matplotlib.pyplot as plt

In [ ]:
np.__version__

In [ ]:
import tensorflow as tf

In [ ]:
tf.__version__

In [ ]:
ROOT_DIR = os.path.abspath('./Mask_RCNN-TF2')
ROOT_DIR

In [ ]:
sys.path

In [ ]:
sys.path.append(ROOT_DIR)

In [ ]:
sys.path

In [ ]:
from mrcnn import utils
from mrcnn import visualize
import mrcnn.model as modellib

In [ ]:
# https://cocodataset.org/#home
sys.path.append(os.path.join(ROOT_DIR, 'samples/coco/'))

In [ ]:
sys.path

In [ ]:
import coco

In [ ]:
MODEL_DIR = os.path.join(ROOT_DIR, 'logs')
IMAGE_DIR = os.path.join(ROOT_DIR, 'images')

In [ ]:
MODEL_DIR, IMAGE_DIR

### [ ! ] Compatibility Update
Run the 5 lines below so we don't have any issues when running with the latest versions of Tensorflow

In [ ]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
config = ConfigProto()
config.gpu_options.allow_growth = True
session = InteractiveSession(config=config)

## Loading the pre-trained neural network

In [ ]:
COCO_MODEL_PATH = os.path.join(ROOT_DIR, 'mask_rcnn_coco.h5')

In [ ]:
utils.download_trained_weights(COCO_MODEL_PATH)

In [ ]:
class InferenceConfig(coco.CocoConfig):
  GPU_COUNT = 1
  IMAGES_PER_GPU = 1

In [ ]:
config = InferenceConfig()

In [ ]:
config.display()

In [ ]:
MODEL_DIR

In [ ]:
network = modellib.MaskRCNN(mode='inference', model_dir=MODEL_DIR, config=config)

In [ ]:
network.load_weights(COCO_MODEL_PATH, by_name=True)

## Detecting objects

In [ ]:
class_names = ['BG', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
               'bus', 'train', 'truck', 'boat', 'traffic light',
               'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird',
               'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear',
               'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie',
               'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball',
               'kite', 'baseball bat', 'baseball glove', 'skateboard',
               'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup',
               'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
               'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
               'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
               'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
               'keyboard', 'cell phone', 'microwave', 'oven', 'toaster',
               'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors',
               'teddy bear', 'hair drier', 'toothbrush']

In [ ]:
len(class_names)

In [ ]:
class_names[1], class_names.index('person')

In [ ]:
image2 = cv2.imread('/content/Mask_RCNN-TF2/images/2516944023_d00345997d_z.jpg')
plt.imshow(image2); # BGR

In [ ]:
image = skimage.io.imread('/content/Mask_RCNN-TF2/images/2516944023_d00345997d_z.jpg') # RGB

In [ ]:
plt.imshow(image);

In [ ]:
class_names[17], class_names[1], class_names[14]

In [ ]:
results = network.detect([image], verbose=0)
results

In [ ]:
r = results[0]

In [ ]:
visualize.display_instances(image, r['rois'], r['masks'],
                            r['class_ids'], class_names, r['scores'])

## Removing the background

In [ ]:
np.unique(r['masks'], return_counts=True)

In [ ]:
r['masks']

In [ ]:
def segment(image, r, index):
  mask = r['masks'][:,:,index]
  #print(mask)
  #print(mask.shape)

  mask = np.stack((mask,) * 3, axis = -1)
  #print(mask)
  #print(mask.shape)

  mask = mask.astype('uint8')
  #print(mask)
  bg = 255 - mask * 255
  #print(mask, mask.min(), mask.max())

  mask_show = np.invert(bg)
  #print(mask_show)
  mask_img = image * mask
  #print(mask_img)

  result = mask_img + bg
  return result, mask_show

In [ ]:
image.shape, 425 * 640

In [ ]:
segmentation, mask_obj = segment(image, r, 0)

In [ ]:
segmentation

In [ ]:
mask_obj

In [ ]:
def show_segment(image, r, index, show_mask = False):
  segmentation, mask_obj = segment(image, r, index)
  plt.subplots(1, figsize=(16,16))
  plt.axis('off')
  if show_mask == True:
    plt.imshow(np.concatenate([mask_obj, segmentation], axis = 1))
  else:
    plt.imshow(np.concatenate([image, segmentation], axis = 1))

In [ ]:
show_segment(image, r, 0, False)

In [ ]:
show_segment(image, r, 0, True)

In [ ]:
r['rois'], len(r['rois'])

In [ ]:
for index in range(len(r['rois'])):
  show_segment(image, r, index, True)

## Segmentation in videos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
capture = cv2.VideoCapture('/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Videos/video_street2.mp4')
connected, frame = capture.read()
connected

In [ ]:
frame.shape

In [ ]:
save_video = cv2.VideoWriter('/content/drive/MyDrive/Cursos - recursos/Computer Vision Masterclass/Videos/video_street2_result_seg.avi',
                             cv2.VideoWriter_fourcc(*'XVID'), 24, (frame.shape[1], frame.shape[0]))

In [ ]:
!cp /content/drive/MyDrive/Cursos\ -\ recursos/Computer\ Vision\ Masterclass/PyCharm/video_functions.py ./Mask_RCNN-TF2/mrcnn

In [ ]:
from mrcnn import video_functions

In [ ]:
colors = video_functions.random_colors(len(class_names), 55)
len(colors)

In [ ]:
print(colors)

In [ ]:
def show(img):
  fig = plt.gcf()
  fig.set_size_inches(16,10)
  plt.axis('off')
  plt.imshow(img)
  plt.show()

In [ ]:
frame_show = 20
current_frame = 0

In [ ]:
while (cv2.waitKey(1) < 0):
  connected, frame = capture.read()

  if not connected:
    break

  results = network.detect([frame], verbose=0)
  r = results[0]

  processed_frame = video_functions.display_instances(frame, r['rois'], r['masks'],
                                                      r['class_ids'], class_names, r['scores'], colors=colors)

  if current_frame <= frame_show:
    show(processed_frame)
    current_frame += 1

  save_video.write(cv2.cvtColor(processed_frame, cv2.COLOR_BGR2RGB))
save_video.release()

## Homework

In [ ]:
images = [os.path.join('/content/Mask_RCNN-TF2/images', f) for f in os.listdir('/content/Mask_RCNN-TF2/images')]
images

In [ ]:
for image in images:
  #try:
  current_image = skimage.io.imread(image)
  (H, W) = current_image.shape[:2]
  #except:
  #  continue

  results = network.detect([current_image], verbose=0)
  r = results[0]
  visualize.display_instances(current_image, r['rois'], r['masks'],
                              r['class_ids'], class_names, r['scores'])